In [ ]:
# Feature Importance (XGBoost) - Note: Only plotting top 20 for visibility since Poly creates 54 features
importances_xgb = xgb.feature_importances_
indices_xgb = np.argsort(importances_xgb)[::-1][:20]  # Grab top 20 features
plt.figure(figsize=(10,6))

In [ ]:
# Handle poly feature names gracefully
poly_feature_names = poly.get_feature_names_out(X.columns)
plt.barh(range(20), importances_xgb[indices_xgb], align='center')
plt.yticks(range(20), [poly_feature_names[i] for i in indices_xgb])
plt.xlabel('Importance')
plt.title('XGBoost Top 20 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Print summary metrics for XGBoost
print("\nXGBoost Model Summary:")
print(f"Accuracy : {xgb_results['accuracy']:.4f}")
print(f"Precision: {xgb_results['precision']:.4f}")
print(f"Recall   : {xgb_results['recall']:.4f}")
print(f"F1-score : {xgb_results['f1']:.4f}")
print(f"ROC-AUC  : {xgb_results['roc_auc']:.4f}")

results_df = pd.DataFrame([lr_results, rf_results, lgbm_results, xgb_results],
                          index=['LogisticRegression', 'RandomForest', 'LightGBM', 'XGBoost'])
print("\nFinal Model Comparison:")
print(results_df.round(4))

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(12,10))
models = [lr, rf, lgbm, xgb]
names = ['Logistic Regression', 'Random Forest', 'LightGBM', 'XGBoost']

for i, (model, name) in enumerate(zip(models, names)):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i//2, i%2],
                xticklabels=['Not Potable', 'Potable'],
                yticklabels=['Not Potable', 'Potable'])
    axes[i//2, i%2].set_title(f'{name}')
    axes[i//2, i%2].set_xlabel('Predicted')
    axes[i//2, i%2].set_ylabel('Actual')
plt.suptitle('Confusion Matrices – All Models', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves for all models
plt.figure(figsize=(10,8))
for model, name in zip(models, names):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves – All Models')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 7. Fine-Tuning XGBoost
print("\n" + "="*60)
print("Starting Hyperparameter Fine-Tuning for All Models...")
print("="*60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tuned_models = {}
tuned_results_list = []

# --- 7.1 Fine-Tune Logistic Regression ---
print("\nFine-tuning Logistic Regression...")
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}
grid_lr = GridSearchCV(LogisticRegression(max_iter=2000, random_state=42),
                       param_grid=param_grid_lr, cv=cv, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X_train, y_train)
best_lr = grid_lr.best_estimator_
tuned_models['Logistic Regression'] = best_lr
tuned_results_list.append(evaluate_model(best_lr, X_test, y_test, "Tuned Logistic Regression"))
